# 05 — Hot-swap timing boundaries

Internal phases, HTTP duration, and sink-observed output gaps remain separate. Queued output can mask internal disruption. N, units, `thesis_evidence=false`, and descriptive-only uncertainty are explicit; missing leaves remain PENDING, never zero.


In [ ]:
import json, os
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
from wafer_analysis.focused import evidence_label, pending_record
from wafer_analysis.paths import resolve_result_batch

def passed_json(batch, artifact):
    rows = []
    for path in sorted(batch.rglob(artifact)):
        status = path.parent / 'canonical-status.json'
        if status.is_file() and json.loads(status.read_text()).get('status') == 'passed':
            rows.append((path, json.loads(path.read_text())))
    return rows

experiments = ('e-swap-1', 'e-swap-2', 'e-swap-4', 'e-swap-6')
records=[]
seen=set()
for experiment in experiments:
    diagnostic_path=os.environ.get(f'{experiment.upper().replace("-", "_")}_DIR')
    if experiment == 'e-swap-1': diagnostic_path=diagnostic_path or os.environ.get('E_SWAP_DIR')
    try:
        batch=resolve_result_batch(experiment, diagnostic_path=diagnostic_path)
    except (FileNotFoundError, RuntimeError, ValueError):
        continue
    for path, evidence in passed_json(batch, 'hotswap-analysis.json'):
        source=evidence.get('measurement_source_leaf')
        if source in seen:
            continue
        seen.add(source)
        for event in evidence.get('events', []):
            records.append({
                'experiment':experiment,
                'condition':evidence.get('condition'),
                'source':source,
                'http_total_ms':event['http_total_ns']/1e6,
                'sink_gap_ms':event['sink_observed_output_gap_ns']/1e6,
            })
if records:
    raw=pd.DataFrame(records)
    independent_runs=raw['source'].nunique()
    summary=(raw.groupby(['experiment','condition'], dropna=False)
        .agg(N_runs=('source','nunique'), N_events=('source','size'),
             median_http_total_ms=('http_total_ms','median'),
             median_sink_gap_ms=('sink_gap_ms','median'))
        .reset_index())
    summary['units']='milliseconds'
    summary['uncertainty']='descriptive only'
    summary['thesis_evidence']=False
    print(f"{evidence_label(independent_runs, 'milliseconds', False)}; independent runs={independent_runs}; events={len(raw)}")
    display(summary)
    fig, ax=plt.subplots()
    for (experiment, condition), group in raw.groupby(['experiment','condition'], dropna=False):
        ax.scatter(group['http_total_ms'],group['sink_gap_ms'],label=f'{experiment}/{condition}')
    ax.set_xscale('log')
    ax.set_yscale('symlog', linthresh=0.01)
    ax.set_xlabel('HTTP duration (ms, log scale)')
    ax.set_ylabel('Sink-observed output gap (ms, symlog scale)')
    ax.set_title(f'Hot-swap timing: N={independent_runs} independent runs, {len(raw)} events (diagnostic)')
    ax.legend()
else:
    display(pd.DataFrame([pending_record('internal and sink-observed hot-swap timing','no passed hotswap-analysis.json leaf','milliseconds')]))
